In [1]:
import pandas as pd
import numpy as np
import glob
import os

In [2]:
instances = [
             'instancia1',
             'instancia2',
             'instancia3',
             'instancia4',
             'instancia5',
             'instancia6',
             'instancia7',
             'instancia8',
             'instancia9',
             'instancia10'
            ]

path = '/home/wilmer/Documentos/Codes/tesis/Resultados/Instancias/'
instance = 'instancia1'

resumo = pd.read_excel('/home/wilmer/Documentos/Codes/tesis/Resultados/Instancias/resumo.xlsx')
resumo['Z_real'] = 0
resumo['Demanda_parametro'] = 0
resumo['Demanda_aproveitada'] = 0
resumo['porcenta_dem_aproveitada'] = 0
resumo['combi_unos'] = 0
resumo['combi_dif_unos'] = 0
resumo['porcenta_combi_dif_unos'] = 0
resumo['clave'] = resumo['Abordagem'] + '_' + resumo['Modelo'] + '_' + resumo['Instância']
resumo['nome_inter'] = resumo['Abordagem'] + '_' + resumo['Modelo']
resumo['Nome_modelo'] = ""

name_model = {
    'GeralModelY_behavioral_base_model':'Model_Basic_Comporta_Est',
    'GeralModelY_behavioral_complete_model':'Model_Comp_Comporta_Est',
    'GeralModelY_behavioral_fullfilment_model':'Model_Fulfill_Comporta_Est',
    'GeralModelY_behavioral_skiplagging_model':'Model_Skipla_Comporta_Est',
    'GeralModelY_independ_base_model':'Model_Basic_Independ_Est',
    'GeralModelY_independ_complete_model':'Model_Comp_Independ_Est',
    'GeralModelY_independ_fullfilment_model':'Model_Fulfill_Independ_Est',
    'GeralModelY_independ_skiplagging_model':'Model_Skipla_Independ_Est',
    'GeralModelYt_behavioral_base_model':'Model_Basic_Comporta_Din',
    'GeralModelYt_behavioral_complete_model':'Model_Comp_Comporta_Din',
    'GeralModelYt_behavioral_fullfilment_model':'Model_Fulfill_Comporta_Din',
    'GeralModelYt_behavioral_skiplagging_model':'Model_Skipla_Comporta_Din',
    'GeralModelYt_independ_base_model':'Model_Basic_Independ_Din',
    'GeralModelYt_independ_complete_model':'Model_Comp_Independ_Din',
    'GeralModelYt_independ_fullfilment_model':'Model_Fulfill_Independ_Din',
    'GeralModelYt_independ_skiplagging_model':'Model_Skipla_Independ_Din'
}


In [3]:
for instance in instances:

    # 2) Consigue todos los archivos con extensión .xls o .xlsx
    excel_patterns = [os.path.join(path + instance, "*.xlsx"), os.path.join(path + instance, "*.xls")]

    excel_files = []
    for pattern in excel_patterns:
        excel_files.extend(glob.glob(pattern))

    for ruta in excel_files:

        data = pd.read_excel(ruta)
        data['Z_real'] = data['Assignments[X]']*data['Preco']

        freq_combinada = (
            data
            .groupby(['Periodo','Vagon','o-d'])
            .size()
            .reset_index(name='Frecuencia')
        )
        
        # calculos
        fo = data['Z_real'].sum()

        dem_total = data['Demanda'].sum()
        x_total = data['Assignments[X]'].sum()

        num_unos = (freq_combinada['Frecuencia'] == 1).sum()
        num_dif_unos = data.shape[0] - num_unos

        # index
        nome_model = os.path.splitext(os.path.basename(ruta))[0]
        indexx = resumo[resumo['clave']==nome_model].index[0]

        # almacenar valor
        resumo.loc[indexx, 'Z_real'] = fo

        resumo.loc[indexx, 'Demanda_parametro'] = dem_total
        resumo.loc[indexx, 'Demanda_aproveitada'] = x_total
        resumo.loc[indexx, 'porcenta_dem_aproveitada'] = round((dem_total-x_total)/dem_total,2)

        resumo.loc[indexx, 'combi_unos'] = num_unos
        resumo.loc[indexx, 'combi_dif_unos'] = num_dif_unos
        resumo.loc[indexx, 'porcenta_combi_dif_unos'] = (num_dif_unos)/data.shape[0]

        nombre_model = resumo.loc[indexx, 'nome_inter']
        resumo.loc[indexx, 'Nome_modelo'] = name_model[nombre_model]


resumo.to_excel('/home/wilmer/Documentos/Codes/tesis/Resultados/Instancias/resumoCor4.xlsx', index=False)

/tmp/ipykernel_20373/2641868281.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '170484.85' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  resumo.loc[indexx, 'Z_real'] = fo
/tmp/ipykernel_20373/2641868281.py:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.01' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  resumo.loc[indexx, 'porcenta_dem_aproveitada'] = round((dem_total-x_total)/dem_total,2)
/tmp/ipykernel_20373/2641868281.py:44: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.07414104882459313' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  resumo.loc[indexx, 'porcenta_combi_dif_unos'] = (num_dif_unos)/data.s

In [4]:
# 1) Datos de ejemplo
df_total = pd.DataFrame(columns=["Instancia", "Nome_modelo", "Nome_modelo_dest", "ptc_variacao"])

for instance in  instances:

    filtro = resumo['Instância'] == instance
    df = resumo[filtro][['Nome_modelo','Z_real']]

    # 2) Convertimos a una Serie indexada por Modelo
    serie = df.set_index('Nome_modelo')['Z_real']

    # 5) (Opcional) Variación porcentual ((i - j) / j * 100)
    matriz_pct = pd.DataFrame(
        (serie.values[:, None] - serie.values[None, :]) / serie.values[:, None] * 100,
        index=serie.index,
        columns=serie.index
    )

    matriz_pct_index = matriz_pct.reset_index()
    resultado_normal = pd.melt(matriz_pct_index, id_vars=[matriz_pct_index.columns[0]], value_vars=matriz_pct_index.columns[1:], var_name='Nome_modelo_dest', value_name='ptc_variacao')
    resultado_normal['Instancia'] = instance
    resultado_normal = resultado_normal[["Instancia", "Nome_modelo", "Nome_modelo_dest", "ptc_variacao"]]

    df_total = pd.concat([df_total, resultado_normal], ignore_index=True)


/tmp/ipykernel_20373/111060035.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_total = pd.concat([df_total, resultado_normal], ignore_index=True)


In [5]:
# df_total.to_excel('/home/wilmer/Documentos/Codes/tesis/Resultados/Instancias/variacoes.xlsx', index=False)